In [ ]:
import numpy as np
import pandas as pd


df_cleaned = pd.read_csv("../../data/processed/balanced_dataset_30k.csv")

In [ ]:
df_cleaned.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df_cleaned["text"]
y = df_cleaned["label"]

X_train, X_test_val, y_train, y_test_val = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_test_val, y_test_val, test_size=0.50, random_state=42, stratify=y_test_val)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=5)
vectorizer.fit(X_train)

In [ ]:
import pickle

with open("../models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [ ]:
X_train_vec = vectorizer.transform(X_train)

In [ ]:
X_test_vec = vectorizer.transform(X_test)
X_val_vec = vectorizer.transform(X_val)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train_vec, y_train)

In [ ]:
model.score(X_test_vec, y_test)

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))

In [ ]:
y_val_pred = model.predict(X_val_vec)
print(classification_report(y_val, y_val_pred))

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc
import seaborn as sns

import matplotlib.pyplot as plt

# Predictions for confusion matrix and ROC
y_pred = model.predict(X_test_vec)
y_score = model.predict_proba(X_test_vec)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Pred 0", "Pred 1"],
    yticklabels=["True 0", "True 1"],
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# ROC curve
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_title("ROC Curve")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
class_0 = df_cleaned[df_cleaned["label"] == 0].sample(24000, random_state=42)
class_1 = df_cleaned[df_cleaned["label"] == 1].sample(6000, random_state=42)

df_distilbert = pd.concat([class_0, class_1]).sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
from sklearn.model_selection import train_test_split

X = df_distilbert["title"]
y = df_distilbert["label"]

X_train_d, X_test_val_d, y_train_d, y_test_val_d = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_test_d, X_val_d, y_test_d, y_val_d = train_test_split(X_test_val_d, y_test_val_d, test_size=0.50, random_state=42, stratify=y_test_val_d)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
result = tokenizer("I feel sad", max_length=20, truncation=True, padding="max_length")
print(result["input_ids"])
print(result["attention_mask"])

In [ ]:
from torch.utils.data import Dataset
from torch import tensor


class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        result = self.tokenizer(self.texts[idx], max_length=512, truncation=True, padding="max_length", return_tensors="pt")

        return {"input_ids": result["input_ids"].squeeze(0), "attention_mask": result["attention_mask"].squeeze(0), "label": tensor(self.labels[idx])}

In [ ]:
train_dataset = DepressionDataset(X_train_d.tolist(), y_train_d.tolist(), tokenizer)
test_dataset = DepressionDataset(X_test_d.tolist(), y_test_d.tolist(), tokenizer)
val_dataset = DepressionDataset(X_val_d.tolist(), y_val_d.tolist(), tokenizer)

In [ ]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
model.to(device)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor([0.625, 2.5]).to(device))

In [ ]:
model.train()
for epoch in range(3):
    total_loss = 0
    batch_num = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        labels = labels.long()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch + 1}, Batch {batch_num + 1}, Loss: {loss.item():.4f}")
        batch_num += 1

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")

In [ ]:
model.eval()
all_preds = []
all_labels = []
batch_num = 0
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        batch_num += 1
        print(f"Validation Batch {batch_num} processed.")
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds))